# Scouting scenarios and validation gates
A club request should define position, season, style, excluded current club, minimum exposure and budget. This notebook distinguishes executable descriptive shortlists from scenarios that cannot be evaluated without historical evidence.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/config.py').exists():
    if ROOT == ROOT.parent: raise RuntimeError('Run inside the project')
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/tfm-matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/tfm-cache')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from src.config import ROOT, MODELS_DIR
from src.preprocessing.clean import load_players
players = load_players()
dataset = json.loads((ROOT / 'reports/dataset.json').read_text())
display(Markdown('**Dataset:** ' + dataset['dataset_kind'] + '. Legacy-labelled observations are preliminary; no synthetic rows are included.'))
assert not players.source.str.contains('demo|synthetic', case=False).any()

from src.models.recommend import PlayerRecommender
rec=PlayerRecommender.load()


**Dataset:** real_legacy_unverified. Legacy-labelled observations are preliminary; no synthetic rows are included.

In [2]:
latest=sorted(rec.players.season.unique())[-1]
requests=[dict(name='Forward scoring shortlist',position='FW',style='finisher',budget_m_eur=10),
          dict(name='Midfield assisting shortlist',position='MF',style='creator',budget_m_eur=5)]
for request in requests:
    display(Markdown('### '+request['name']))
    display(rec.recommend(request['style'],position=request['position'],season=latest,exclude_club='Barcelona',top_n=5)[['player','team','minutes','gls_p90','ast_p90','affinity']])
    decision_date=latest[5:]+'-07-01'
    try:
        budgeted=rec.recommend(request['style'],position=request['position'],season=latest,
                    max_budget=request['budget_m_eur'],cutoff=decision_date)
        display(budgeted)
    except ValueError as exc:
        display(Markdown('**Budget scenario blocked:** '+str(exc)))


### Forward scoring shortlist

,player,team,minutes,gls_p90,ast_p90,affinity
345,Umar Sadiq,Real Sociedad,1077.0,0.250696,0.083565,0.999505
215,Dodi Lukebakio,Sevilla,1125.0,0.400000,0.080000,0.999361
99,Anastasios Douvikas,Celta Vigo,905.0,0.696133,0.099448,0.999266
52,Ante Budimir,Osasuna,2435.0,0.628337,0.073922,0.999220
369,Jørgen Strand Larsen,Celta Vigo,2863.0,0.408662,0.094307,0.998899


**Budget scenario blocked:** Budget requires dated EUR valuations and a decision cutoff; unknown values are excluded

### Midfield assisting shortlist

,player,team,minutes,gls_p90,ast_p90,affinity
16,Hugo Álvarez,Celta Vigo,762.0,0.118110,0.118110,0.999989
4,Ilias Akhomach,Villarreal,1511.0,0.119126,0.119126,0.999901
66,William Carvalho,Real Betis,680.0,0.132353,0.397059,0.999631
251,Luka Modrić,Real Madrid,1688.0,0.106635,0.319905,0.999543
299,Carles Pérez,Celta Vigo,772.0,0.116580,0.116580,0.999384


**Budget scenario blocked:** Budget requires dated EUR valuations and a decision cutoff; unknown values are excluded

These are documented analysis requests executed against legacy football observations, not actual club mandates. Unknown valuations are never treated as free transfers; market value is not the same as an agreed fee or total recruitment cost. A 2026 current valuation cannot be attached to a 2024 shortlist.

In [3]:
from src.data.matching import propose_matches
profile_path=ROOT/'data/external/transfermarkt/current_profiles_sample.csv'
if profile_path.exists() and profile_path.stat().st_size>1:
    tm=pd.read_csv(profile_path,dtype={'source_player_id':str})
    entities=players.drop_duplicates('source_player_id')[['source_player_id','player']]
    proposals=propose_matches(entities,tm)
    display(proposals.status.value_counts().rename('identities').to_frame())
    display(proposals[proposals.status!='unmatched'].head(20))
    proposals.to_csv(ROOT/'reports/identity_proposals.csv',index=False)
else:
    display(Markdown('Profile identity matching unavailable: no parsed profile sample.'))


,identities
status,
unmatched,417
review_required,2


,left_id,right_id,status,candidate_ids
321,legacy:2fd50d0c0d7bca8e,None,review_required,527966
412,legacy:52cb3b1412d45945,None,review_required,937958


No cross-source identity is auto-approved from a name alone. Unique name plus date of birth or an explicit reviewed mapping is required. Current sampled profiles do not provide historical club membership. Retrospective transfer evaluation needs a dated candidate universe, prior-only feature fitting, historical valuation knowledge dates, transfer dates and actual fees. None is reconstructed from future observations.

In [4]:
requirements=pd.DataFrame([
 {'requirement':'Multiple consecutive statistical seasons','status':'blocked' if players.season.nunique()<2 else 'available'},
 {'requirement':'Historical dated candidate roster','status':'blocked'},
 {'requirement':'As-of EUR valuations / actual transfer fees','status':'blocked'},
 {'requirement':'Verified identity-to-transfer labels','status':'blocked'},
 {'requirement':'Recommendation engine and cutoff guard','status':'executed in automated tests'},
])
display(requirements)


,requirement,status
0,Multiple consecutive statistical seasons,blocked
1,Historical dated candidate roster,blocked
2,As-of EUR valuations / actual transfer fees,blocked
3,Verified identity-to-transfer labels,blocked
4,Recommendation engine and cutoff guard,executed in automated tests


## Required historical experiment
For each summer decision date, fit imputation, scaling, PCA and clustering only on completed prior seasons. Freeze the eligibility universe as known then. Compare position/budget-eligible ranked shortlists with observed next-window transfers, with recall@k, precision@k and NDCG only when relevance labels and sampling rules are documented. Report selection and survivorship bias. This design is documented, but transfer metrics remain unrun because the required labels are missing.